# Emotion classifier and Qwen review

Open this notebook in Google Colab and run the cells in order. The last cell opens the app inside the notebook. Enter text and select **Analyze text** to see the three labels reviewed by Qwen.

The notebook needs your fine-tuned DeBERTa checkpoint folder and an OpenRouter API key. In Colab’s **Secrets** panel, add `OPENROUTER_API_KEY` and enable notebook access. You can also set the environment variable or enter the key at the hidden prompt.

Each analysis sends the submitted text and three candidate labels to OpenRouter. Vivid color shows **High** confidence from `AGREE`, a muted tint shows **Low** confidence from `DISAGREE`, and labels outside the top three stay gray. Hover, focus, or tap a colored label to inspect the verdict.

## Install dependencies

This uses Colab’s PyTorch installation. The classifier uses a GPU when Colab provides one and can also run on CPU.

In [45]:
%pip install -q "transformers>=5,<6" sentencepiece "httpx==0.28.1" "jsonschema>=4.18,<5"

## Load the nine-label classifier

Set `CLASSIFIER_MODEL_DIR` to the fine-tuned model folder that contains `config.json`, the model weights, and the saved tokenizer files. The default points to the Google Drive folder used in the supplied inference notebook. If you copied the model into the Colab runtime, use that folder's path instead.

The pipeline ranks all nine labels by their unrounded sigmoid scores, including Neutral. The three highest-scoring labels go to Qwen in descending order. The checkpoint's label order breaks ties.

In [46]:
from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

CLASSIFIER_MODEL_DIR = Path("/content/drive/MyDrive/COSC 72/model_v1")
CLASSIFIER_NORMALIZE = False
CLASSIFIER_SPLIT_LONG_TEXTS = False
CLASSIFIER_BATCH_SIZE = 64

if CLASSIFIER_MODEL_DIR.is_relative_to("/content/drive") and not CLASSIFIER_MODEL_DIR.is_dir():
    from google.colab import drive

    drive.mount("/content/drive")

if not (CLASSIFIER_MODEL_DIR / "config.json").is_file():
    raise FileNotFoundError(
        f"Model config is missing in {CLASSIFIER_MODEL_DIR}. "
        "Set CLASSIFIER_MODEL_DIR to the fine-tuned checkpoint folder "
        "containing config.json, the weights, and the tokenizer files."
    )

CLASSIFIER_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLASSIFIER_USE_BF16 = CLASSIFIER_DEVICE == "cuda" and torch.cuda.is_bf16_supported(
    including_emulation=False
)
classifier_tokenizer = AutoTokenizer.from_pretrained(
    CLASSIFIER_MODEL_DIR, use_fast=False, local_files_only=True
)
classifier_model, classifier_load_info = AutoModelForSequenceClassification.from_pretrained(
    CLASSIFIER_MODEL_DIR, local_files_only=True, output_loading_info=True
)
if any(
    key.startswith(("classifier.", "pooler."))
    for key in classifier_load_info["missing_keys"]
):
    raise ValueError("The checkpoint is missing fine-tuned classifier weights.")

CLASSIFIER_LABELS = tuple(
    classifier_model.config.id2label[index]
    for index in range(classifier_model.config.num_labels)
)
EXPECTED_CLASSIFIER_LABELS = {
    "anger", "anticipation", "disgust", "fear", "joy",
    "neutral", "sadness", "surprise", "trust",
}
if (
    len(CLASSIFIER_LABELS) != len(EXPECTED_CLASSIFIER_LABELS)
    or {label.casefold() for label in CLASSIFIER_LABELS} != EXPECTED_CLASSIFIER_LABELS
):
    raise ValueError(f"Expected the project's nine emotion labels. Found {CLASSIFIER_LABELS!r}.")

CLASSIFIER_NEUTRAL_INDEX = next(
    index for index, label in enumerate(CLASSIFIER_LABELS) if label.casefold() == "neutral"
)
tokenizer_max_length = classifier_tokenizer.model_max_length
CLASSIFIER_MAX_LENGTH = min(
    tokenizer_max_length if tokenizer_max_length < 100_000 else 512,
    classifier_model.config.max_position_embeddings,
)
classifier_model = classifier_model.to(CLASSIFIER_DEVICE).eval()

print(f"Classifier loaded on {CLASSIFIER_DEVICE} with a {CLASSIFIER_MAX_LENGTH}-token limit.")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Classifier loaded on cuda with a 128-token limit.


By default the classifier scores a single sequence and truncates it at the checkpoint’s token limit. Set `CLASSIFIER_SPLIT_LONG_TEXTS = True` to score each sentence, then take the maximum for each emotion and the minimum for Neutral. Qwen receives the original submitted text.

In [47]:
import re

CLASSIFIER_URL_RE = re.compile(r"https?://\S+|www\.\S+")
CLASSIFIER_MENTION_RE = re.compile(r"@\w+")
CLASSIFIER_SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")


def classifier_top_three(text: str) -> list[str]:
    """Return the three highest-scoring labels in checkpoint spelling."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Enter text to classify.")

    classifier_text = text.strip()
    if CLASSIFIER_NORMALIZE:
        classifier_text = CLASSIFIER_MENTION_RE.sub("@user", classifier_text)
        classifier_text = CLASSIFIER_URL_RE.sub("http", classifier_text)

    pieces = CLASSIFIER_SENTENCE_RE.split(classifier_text) if CLASSIFIER_SPLIT_LONG_TEXTS else [classifier_text]
    probability_batches = []
    with torch.inference_mode():
        for start in range(0, len(pieces), CLASSIFIER_BATCH_SIZE):
            batch = pieces[start:start + CLASSIFIER_BATCH_SIZE]
            encoded = classifier_tokenizer(
                batch, truncation=True, max_length=CLASSIFIER_MAX_LENGTH,
                padding=True, return_tensors="pt",
            ).to(CLASSIFIER_DEVICE)
            with torch.autocast(
                device_type=CLASSIFIER_DEVICE,
                dtype=torch.bfloat16,
                enabled=CLASSIFIER_USE_BF16,
            ):
                logits = classifier_model(**encoded).logits
            probability_batches.append(logits.float().sigmoid().cpu())

        probabilities = torch.cat(probability_batches, dim=0)
        scores = probabilities.amax(dim=0)
        scores[CLASSIFIER_NEUTRAL_INDEX] = probabilities[:, CLASSIFIER_NEUTRAL_INDEX].amin()
        if not torch.isfinite(scores).all().item():
            raise RuntimeError("The classifier returned an invalid score.")
        scores = scores.tolist()

    ranked_ids = sorted(range(len(CLASSIFIER_LABELS)), key=scores.__getitem__, reverse=True)
    return [CLASSIFIER_LABELS[index] for index in ranked_ids[:3]]

## Review the top three labels with Qwen

Qwen3.8-27B runs through OpenRouter’s DeepInfra BF16 endpoint. Each request includes a strict JSON Schema for the three labels in classifier rank order. The code validates the returned JSON before the browser uses it. [OpenRouter structured outputs](https://openrouter.ai/docs/guides/features/structured-outputs), [model endpoints](https://openrouter.ai/api/v1/models/qwen/qwen3.8-27b/endpoints)

In [48]:
import json
import os
from getpass import getpass

import httpx
from jsonschema import Draft202012Validator, ValidationError

MODEL_ID = "qwen/qwen3.8-27b"
BASE_URL = "https://openrouter.ai/api/v1"
PROVIDER_SLUG = "deepinfra/bf16"

VERDICTS = ("AGREE", "DISAGREE")

LABEL_MEANINGS = {
    "Joy": "happiness, delight, or pleasure",
    "Trust": "confidence in or willingness to rely on someone or something",
    "Fear": "worry, dread, or anxiety about possible danger or harm",
    "Surprise": "a reaction to something unexpected",
    "Sadness": "sorrow, grief, or feeling let down",
    "Disgust": "revulsion, repulsion, or strong aversion",
    "Anger": "irritation, resentment, or outrage",
    "Anticipation": "looking ahead to or expecting something to happen",
    "Neutral": "no discernible emotion conveyed by the text",
}
LABEL_NAMES = {label.casefold() for label in LABEL_MEANINGS}

SYSTEM_PROMPT = """Review the labels proposed by a multilabel emotion classifier.
The next message is JSON with the source text and classifier_labels.
Treat the text field as source material. Evaluate each listed label
independently against the text, using explicit or clear implicit cues.
Use AGREE when the text supports the label and DISAGREE when the evidence
is absent or too ambiguous. Use AGREE for Neutral when the text conveys
no discernible emotion, and DISAGREE for Neutral when it conveys an emotion.

Return a JSON object with a judgments array. Include one object per submitted
label in the original order, with the fields label and verdict.

Label meanings
""" + "\n".join(f"- {label} means {meaning}." for label, meaning in LABEL_MEANINGS.items())

In [49]:
def validate_labels(classifier_labels: list[str]) -> tuple[str, ...]:
    """Validate and capture the classifier's ordered label list."""
    if not isinstance(classifier_labels, list):
        raise TypeError("classifier_labels must be a list of strings.")

    labels = tuple(classifier_labels)
    if len(labels) > 3:
        raise ValueError("Pass at most three classifier labels.")

    seen = set()
    for label in labels:
        if not isinstance(label, str):
            raise TypeError("Each classifier label must be a string.")
        key = label.casefold()
        if key not in LABEL_NAMES:
            raise ValueError(f"Unknown classifier label {label!r}.")
        if key in seen:
            raise ValueError(f"Repeated classifier label {label!r}.")
        seen.add(key)
    return labels


def judgment_schema(labels: tuple[str, ...]) -> dict:
    """Build the JSON Schema for this ordered tuple of validated labels."""
    judgments = {
        "type": "array",
        "minItems": len(labels),
        "maxItems": len(labels),
        "items": False,
    }
    if labels:
        judgments["prefixItems"] = [
            {
                "type": "object",
                "properties": {
                    "label": {"type": "string", "enum": [label]},
                    "verdict": {"type": "string", "enum": list(VERDICTS)},
                },
                "required": ["label", "verdict"],
                "additionalProperties": False,
            }
            for label in labels
        ]

    return {
        "type": "object",
        "properties": {"judgments": judgments},
        "required": ["judgments"],
        "additionalProperties": False,
    }


def unique_json_object(pairs: list[tuple[str, object]]) -> dict:
    """Reject repeated fields when parsing JSON."""
    result = dict(pairs)
    if len(result) != len(pairs):
        raise ValueError("The JSON object contains duplicate fields.")
    return result


def judge_emotions(
    text: str, classifier_labels: list[str], *, client: httpx.Client
) -> str:
    """Return Qwen's constrained JSON, or raise if generation fails."""
    if not isinstance(text, str):
        raise TypeError("text must be the original input as a string.")

    labels = validate_labels(classifier_labels)
    if not labels:
        return '{"judgments":[]}'

    schema = judgment_schema(labels)
    input_json = json.dumps(
        {"text": text, "classifier_labels": labels}, ensure_ascii=False
    )
    # Escape literal chat control markers inside the source text.
    input_json = input_json.replace("<|", r"\u003c|")

    response = client.post(
        "chat/completions",
        json={
            "model": MODEL_ID,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": input_json},
            ],
            "stream": False,
            "temperature": 0.0,
            "max_tokens": 256,
            "reasoning": {"enabled": False},
            "provider": {"only": [PROVIDER_SLUG], "require_parameters": True},
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": "emotion_judgments",
                    "strict": True,
                    "schema": schema,
                },
            },
        },
    )

    try:
        body = response.json()
    except ValueError as exc:
        response.raise_for_status()
        raise RuntimeError("OpenRouter returned an invalid API response.") from exc

    # OpenRouter can report an upstream error in an HTTP 200 response.
    error = body.get("error") if isinstance(body, dict) else None
    if isinstance(error, dict):
        error_code = error.get("code", response.status_code)
        error_message = error.get("message", "Request failed.")
        raise RuntimeError(f"OpenRouter API error {error_code}. {error_message}")
    response.raise_for_status()

    try:
        completion = body["choices"][0]
        finish_reason = completion["finish_reason"]
        output = completion["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise RuntimeError("Invalid chat completion response from OpenRouter.") from exc

    if finish_reason != "stop":
        raise RuntimeError(f"Expected finish_reason 'stop', received {finish_reason!r}.")

    try:
        parsed = json.loads(output, object_pairs_hook=unique_json_object)
        Draft202012Validator(schema).validate(parsed)
    except (TypeError, ValueError, ValidationError) as exc:
        raise RuntimeError("Completion fell outside the requested JSON schema.") from exc

    return output

## Web interface

The page posts the source text to the notebook’s server, draws the returned judgments, and lets you inspect the Qwen JSON.

In [50]:
# @title Emotion wheel page
PAGE_HTML = r'''<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Plutchik emotion wheel</title>
  <style>
    :root {
      color-scheme: light dark;
      --page: #f6f5ef;
      --paper: #fffefa;
      --ink: #20251e;
      --muted: #595f53;
      --rule: #dedfd4;
      --action: #283d32;
      --action-text: #ffffff;
      --focus: #366e56;
      --gray-slice: #e2e2df;
      --gray-hub: #d8d8d5;
      --wheel-rim: #a4a49c;
      --danger: #942d29;
      --tooltip: #26332a;
      --tooltip-ink: #ffffff;
      font-family: "DM Sans", system-ui, sans-serif;
    }
    @media (prefers-color-scheme: dark) {
      :root {
        --page: #171c18;
        --paper: #1e2520;
        --ink: #edf0e8;
        --muted: #b9c2b5;
        --rule: #39453b;
        --action: #cee2d3;
        --action-text: #1d3025;
        --focus: #a4d6b6;
        --gray-slice: #343936;
        --gray-hub: #3f4541;
        --wheel-rim: #727c73;
        --danger: #ffaba2;
        --tooltip: #dce8df;
        --tooltip-ink: #1d3025;
      }
    }
    *, *::before, *::after { box-sizing: border-box; }
    [hidden] { display: none !important; }
    body { margin: 0; min-width: 0; color: var(--ink); background: var(--page); }
    button, textarea { font: inherit; }
    button:focus-visible, textarea:focus-visible, summary:focus-visible {
      outline: 2px solid var(--focus);
      outline-offset: 3px;
    }
    .page { max-width: 1160px; margin: 0 auto; padding: clamp(22px, 4vw, 54px); }
    .page-heading {
      border-top: 1px solid var(--rule);
      padding-top: clamp(16px, 2.6vw, 32px);
      margin-bottom: clamp(20px, 3vw, 34px);
    }
    h1 {
      font-family: "Instrument Serif", Georgia, serif;
      font-size: clamp(2.05rem, 5.4vw, 4.25rem);
      font-weight: 400;
      line-height: 1.05;
      letter-spacing: -.04em;
      margin: 0 0 .3em;
    }
    .subtitle { margin: 0; color: var(--muted); line-height: 1.55; }
    .workspace {
      display: grid;
      grid-template-columns: minmax(240px, 4fr) minmax(0, 7fr);
      gap: clamp(14px, 3vw, 34px);
      align-items: start;
    }
    .input-panel, .wheel-panel {
      min-width: 0;
      border: 1px solid var(--rule);
      border-radius: 16px;
      margin: 0;
      background: var(--paper);
    }
    .input-panel { padding: clamp(18px, 3vw, 30px); }
    .input-label {
      display: block;
      margin-bottom: 10px;
      font-size: 1.1rem;
      font-weight: 700;
    }
    textarea {
      display: block;
      width: 100%;
      min-height: 10rem;
      margin: 0;
      padding: 13px 14px;
      resize: vertical;
      color: var(--ink);
      background: var(--page);
      border: 1px solid var(--rule);
      border-radius: 9px;
      font-size: 1rem;
      line-height: 1.5;
    }
    textarea::placeholder { color: var(--muted); }
    .analyze {
      width: 100%;
      min-height: 46px;
      margin-top: 14px;
      padding: 10px 16px;
      border: 1px solid transparent;
      border-radius: 8px;
      color: var(--action-text);
      background: var(--action);
      font-weight: 700;
      cursor: pointer;
    }
    .analyze:not(:disabled):hover { filter: brightness(1.13); }
    .analyze:disabled { cursor: default; opacity: .65; }
    .status, .error { margin: 13px 0 0; font-size: .875rem; line-height: 1.5; }
    .status { color: var(--muted); }
    .error { color: var(--danger); overflow-wrap: anywhere; }
    .status:empty, .error:empty { margin: 0; }
    details {
      border-top: 1px solid var(--rule);
      margin-top: 20px;
      padding-top: 13px;
      font-size: .875rem;
    }
    summary { color: var(--ink); cursor: pointer; font-weight: 650; }
    pre {
      max-height: 17rem;
      overflow: auto;
      margin: 12px 0 0;
      padding: 12px;
      color: var(--ink);
      background: var(--page);
      border-radius: 8px;
      white-space: pre-wrap;
      overflow-wrap: anywhere;
      line-height: 1.4;
    }
    .wheel-panel {
      position: relative;
      display: grid;
      justify-items: center;
      gap: 18px;
      padding: clamp(10px, 2.7vw, 30px);
    }
    .wheel {
      position: relative;
      width: 100%;
      max-width: 474px;
      aspect-ratio: 1;
      container-type: inline-size;
    }
    .slice {
      appearance: none;
      position: absolute;
      border: 0;
      padding: 0;
      background: transparent;
      color: var(--ink);
      --idle-fill: var(--gray-slice);
      --agree-fill: hsl(var(--hue) 86% var(--lightness));
    }
    @media (prefers-color-scheme: dark) {
      .slice { --agree-fill: hsl(var(--hue) 78% var(--dark-lightness)); }
    }
    .slice[data-emotion="neutral"] { --idle-fill: var(--gray-hub); }
    .slice:disabled { opacity: 1; cursor: default; }
    .slice:not(:disabled) { cursor: help; }
    .slice::before {
      position: absolute;
      inset: 0;
      content: "";
      background: var(--idle-fill);
      filter: grayscale(1);
      transition: background-color 170ms ease, filter 170ms ease;
    }
    .slice[data-verdict="AGREE"]::before { background: var(--agree-fill); filter: none; }
    .slice[data-verdict="DISAGREE"]::before {
      background: color-mix(in srgb, var(--agree-fill) 31%, var(--idle-fill));
      filter: none;
    }
    .slice-label {
      position: absolute;
      transform: translate(-50%, -50%);
      font-size: clamp(.75rem, 3.05cqi, .95rem);
      font-weight: 650;
      line-height: 1.1;
      white-space: nowrap;
      pointer-events: none;
    }
    .slice[data-emotion="neutral"] .slice-label { font-size: clamp(1.05rem, 4.5cqi, 1.35rem); }
    .wheel[data-compact] [data-emotion="anticipation"] .slice-label {
      max-width: 4.7em;
      white-space: normal;
    }
    .slice:focus-visible .slice-label { outline: 2px solid var(--focus); outline-offset: 2px; }
    .slice:not(:disabled):hover .slice-label { text-decoration: underline; text-underline-offset: .2em; }
    .slice-symbol {
      position: absolute;
      top: calc(100% + .18em);
      left: 50%;
      transform: translateX(-50%);
      font-size: .78em;
    }
    .rim, .hub-rim {
      position: absolute;
      z-index: 1;
      border-radius: 50%;
      border: 1px solid var(--wheel-rim);
      pointer-events: none;
    }
    .rim { inset: .4%; }
    .hub-rim { inset: var(--hub-inset, 17.25%); }
    .key {
      display: flex;
      flex-wrap: wrap;
      justify-content: center;
      gap: 8px 16px;
      color: var(--muted);
      font-size: .81rem;
    }
    .key-icon {
      display: inline-block;
      min-width: 1em;
      color: var(--ink);
      font-weight: 750;
      text-align: center;
    }
    .wheel-tooltip {
      position: absolute;
      z-index: 3;
      max-width: calc(100% - 12px);
      padding: 7px 10px;
      border-radius: 7px;
      background: var(--tooltip);
      color: var(--tooltip-ink);
      font-size: .82rem;
      line-height: 1.3;
      box-shadow: 0 3px 9px #0003;
      pointer-events: none;
    }
    @media (max-width: 680px) {
      .workspace { grid-template-columns: minmax(0, 1fr); }
      .page { padding: 18px; }
      textarea { min-height: 8rem; }
    }
    @media (prefers-reduced-motion: reduce) {
      .slice::before { transition: none; }
    }
  </style>
</head>
<body>
  <main class="page">
    <header class="page-heading">
      <h1>Plutchik emotion wheel</h1>
      <p class="subtitle">See how Qwen judges the classifier’s three highest-scoring labels.</p>
    </header>
    <div class="workspace">
      <form class="input-panel" id="analysis-form">
        <label class="input-label" for="source-text">Text to analyze</label>
        <textarea id="source-text" name="text" rows="5" required
          placeholder="Enter or paste the text you want to evaluate."
          aria-errormessage="analysis-error"></textarea>
        <button class="analyze" id="analyze" type="submit" disabled>Analyze text</button>
        <p class="status" id="analysis-status" role="status"></p>
        <p class="error" id="analysis-error" role="alert" hidden></p>
        <details id="result-details" hidden>
          <summary>Qwen JSON</summary>
          <pre id="result-json"></pre>
        </details>
      </form>
      <figure class="wheel-panel" id="wheel-panel">
        <div class="wheel" id="wheel" role="group" aria-label="Eight emotion slices around Neutral">
          <span class="rim" aria-hidden="true"></span>
          <span class="hub-rim" aria-hidden="true"></span>
        </div>
        <div class="wheel-tooltip" id="wheel-tooltip" aria-hidden="true" hidden></div>
        <figcaption class="key">
          <span><span class="key-icon" aria-hidden="true">✓</span> High confidence</span>
          <span><span class="key-icon" aria-hidden="true">−</span> Low confidence</span>
          <span><span class="key-icon" aria-hidden="true">○</span> Unreviewed</span>
        </figcaption>
      </figure>
    </div>
  </main>
  <script>
    (() => {
      "use strict";

      const form = document.getElementById("analysis-form");
      const input = document.getElementById("source-text");
      const analyze = document.getElementById("analyze");
      const status = document.getElementById("analysis-status");
      const error = document.getElementById("analysis-error");
      const resultDetails = document.getElementById("result-details");
      const resultJson = document.getElementById("result-json");
      const wheel = document.getElementById("wheel");
      const panel = document.getElementById("wheel-panel");
      const tooltip = document.getElementById("wheel-tooltip");
      // Order and hues follow the previous wheel, with Neutral at the center.
      const palette = [
        ["Joy", 56, 71, 27],
        ["Trust", 105, 74, 28],
        ["Fear", 132, 58, 23],
        ["Surprise", 203, 73, 34],
        ["Sadness", 240, 77, 39],
        ["Disgust", 300, 71, 33],
        ["Anger", 0, 69, 33],
        ["Anticipation", 30, 71, 28],
        ["Neutral", 177, 77, 27],
      ];
      const verdicts = new Map([
        ["AGREE", { confidence: "High", symbol: "✓", description: "Qwen agrees" }],
        ["DISAGREE", { confidence: "Low", symbol: "−", description: "Qwen disagrees" }],
      ]);
      const slices = new Map();
      const outerRadius = 49.6;
      let compactLayout;
      let pending = null;
      let hovered = null;
      let focused = null;
      let pinned = null;

      function point(radius, angle) {
        const radians = angle * Math.PI / 180;
        return [50 + radius * Math.cos(radians), 50 + radius * Math.sin(radians)];
      }

      function arc(radius, start, end) {
        return Array.from({ length: 21 }, (_, step) =>
          point(radius, start + (end - start) * step / 20));
      }

      function setBounds(button, left, top, width, height) {
        Object.assign(button.style, {
          left: left + "%", top: top + "%", width: width + "%", height: height + "%",
        });
      }

      function placeSlice({ button, label, index }, innerRadius, halfAngle) {
        if (index === 8) {
          setBounds(button, 50 - innerRadius, 50 - innerRadius, 2 * innerRadius, 2 * innerRadius);
          button.style.clipPath = "circle(50%)";
          label.style.left = "50%";
          label.style.top = "50%";
          return;
        }
        const angle = -90 + index * 45;
        const points = [
          ...arc(outerRadius, angle - halfAngle, angle + halfAngle),
          ...arc(innerRadius, angle + 22.5, angle - 22.5),
        ];
        const xs = points.map(([x]) => x);
        const ys = points.map(([, y]) => y);
        const left = Math.min(...xs);
        const top = Math.min(...ys);
        const width = Math.max(...xs) - left;
        const height = Math.max(...ys) - top;
        const percent = (value, start, length) => ((value - start) / length * 100).toFixed(4) + "%";
        const vertices = points.map(([x, y]) =>
          percent(x, left, width) + " " + percent(y, top, height));
        const [labelX, labelY] = point((outerRadius + innerRadius) / 2, angle);
        setBounds(button, left, top, width, height);
        button.style.clipPath = "polygon(" + vertices.join(",") + ")";
        label.style.left = percent(labelX, left, width);
        label.style.top = percent(labelY, top, height);
      }

      function updateTooltip() {
        const slice = hovered || pinned || focused;
        if (!slice || !slice.review) {
          tooltip.hidden = true;
          return;
        }
        tooltip.textContent = slice.name + " · " + slice.review.confidence + " confidence";
        tooltip.hidden = false;
        const panelRect = panel.getBoundingClientRect();
        const labelRect = slice.label.getBoundingClientRect();
        const center = (labelRect.left + labelRect.right) / 2 - panelRect.left;
        const inset = 6;
        const left = Math.max(inset, Math.min(center - tooltip.offsetWidth / 2,
          panelRect.width - tooltip.offsetWidth - inset));
        const above = labelRect.top - panelRect.top - tooltip.offsetHeight - 10;
        const top = above >= inset ? above : labelRect.bottom - panelRect.top + 10;
        tooltip.style.left = left + "px";
        tooltip.style.top = top + "px";
      }

      function closeTooltip() {
        hovered = null;
        focused = null;
        pinned = null;
        tooltip.hidden = true;
      }

      for (const [index, [name, hue, lightness, darkLightness]] of palette.entries()) {
        const key = name.toLowerCase();
        const button = document.createElement("button");
        const label = document.createElement("span");
        const symbol = document.createElement("span");
        const slice = { name, index, button, label, symbol, review: null };
        button.type = "button";
        button.className = "slice";
        button.dataset.emotion = key;
        button.style.setProperty("--hue", String(hue));
        button.style.setProperty("--lightness", lightness + "%");
        button.style.setProperty("--dark-lightness", darkLightness + "%");
        label.className = "slice-label";
        label.textContent = name === "Anticipation" ? "Antici\u00adpation" : name;
        symbol.className = "slice-symbol";
        symbol.setAttribute("aria-hidden", "true");
        label.append(symbol);
        button.append(label);
        wheel.append(button);
        slices.set(key, slice);

        button.addEventListener("pointerenter", event => {
          if (event.pointerType !== "touch" && slice.review) {
            hovered = slice;
            updateTooltip();
          }
        });
        button.addEventListener("pointerleave", () => {
          if (hovered === slice) hovered = null;
          updateTooltip();
        });
        button.addEventListener("focus", () => {
          if (button.matches(":focus-visible")) focused = slice;
          updateTooltip();
        });
        button.addEventListener("blur", () => {
          if (focused === slice) focused = null;
          updateTooltip();
        });
        button.addEventListener("click", () => {
          if (!slice.review) return;
          pinned = pinned === slice ? null : slice;
          updateTooltip();
        });
      }

      function fitWheel() {
        const compact = wheel.getBoundingClientRect().width < 416;
        if (compact !== compactLayout) {
          compactLayout = compact;
          wheel.toggleAttribute("data-compact", compact);
          const innerRadius = compact ? 26 : 32.75;
          const halfAngle = compact ? 17 : 15;
          wheel.style.setProperty("--hub-inset", (50 - innerRadius) + "%");
          for (const slice of slices.values()) placeSlice(slice, innerRadius, halfAngle);
        }
        if (!tooltip.hidden) updateTooltip();
      }

      fitWheel();
      new ResizeObserver(fitWheel).observe(wheel);
      document.addEventListener("keydown", event => {
        if (event.key === "Escape") closeTooltip();
      });
      document.addEventListener("pointerdown", event => {
        if (!wheel.contains(event.target)) closeTooltip();
      });

      function paint(judgments) {
        closeTooltip();
        const reviewed = new Map(judgments.map(({ label, verdict }) => [
          label.normalize("NFKC").toLowerCase(), verdict,
        ]));
        for (const [key, slice] of slices) {
          const verdict = reviewed.get(key);
          slice.review = verdicts.get(verdict);
          slice.button.dataset.verdict = verdict || "unreviewed";
          slice.button.disabled = !slice.review;
          slice.symbol.textContent = slice.review?.symbol || "";
          const description = slice.review
            ? slice.review.confidence + " confidence. " + slice.review.description + "."
            : "Unreviewed.";
          slice.button.setAttribute("aria-label", slice.name + ". " + description);
        }
      }

      function setBusy(busy) {
        wheel.setAttribute("aria-busy", String(busy));
        analyze.disabled = busy || !input.value.trim();
        analyze.textContent = busy ? "Analyzing…" : "Analyze text";
      }

      function clearResult() {
        paint([]);
        resultDetails.hidden = true;
        resultDetails.open = false;
        resultJson.textContent = "";
        error.hidden = true;
        error.textContent = "";
        status.textContent = "";
        input.removeAttribute("aria-invalid");
      }

      function showError(message) {
        status.textContent = "";
        error.textContent = message;
        error.hidden = false;
      }

      function validateJudgments(payload) {
        const problem = "The server returned an invalid judgment response. Please try again.";
        if (!Array.isArray(payload?.judgments) || payload.judgments.length > 3) {
          throw new Error(problem);
        }
        const seen = new Set();
        for (const result of payload.judgments) {
          if (typeof result?.label !== "string" || !verdicts.has(result?.verdict)) {
            throw new Error(problem);
          }
          const key = result.label.normalize("NFKC").toLowerCase();
          if (!slices.has(key) || seen.has(key)) throw new Error(problem);
          seen.add(key);
        }
        return payload.judgments;
      }

      input.addEventListener("input", () => {
        pending?.abort();
        pending = null;
        clearResult();
        setBusy(false);
      });

      form.addEventListener("submit", async event => {
        event.preventDefault();
        const text = input.value;
        if (pending) return;
        clearResult();
        if (!text.trim()) {
          input.setAttribute("aria-invalid", "true");
          showError("Enter some text to analyze.");
          input.focus();
          return;
        }

        const request = new AbortController();
        pending = request;
        setBusy(true);
        status.textContent = "Classifying text and requesting Qwen’s judgments…";
        try {
          const response = await fetch("./api/analyze", {
            method: "POST",
            headers: { "Content-Type": "application/json" },
            body: JSON.stringify({ text }),
            signal: request.signal,
          });
          const payload = await response.json().catch(() => null);
          if (pending !== request) return;
          if (!response.ok) {
            throw new Error(typeof payload?.error === "string"
              ? payload.error : "The analysis request failed. Please try again.");
          }
          const judgments = validateJudgments(payload);
          paint(judgments);
          resultJson.textContent = JSON.stringify({ judgments }, null, 2);
          resultDetails.hidden = false;
          status.textContent = judgments.length
            ? "Hover, focus, or tap a colored slice to see its confidence."
            : "Qwen returned no judgments.";
        } catch (problem) {
          if (pending === request && problem.name !== "AbortError") {
            showError(problem instanceof TypeError
              ? "The Colab server could not be reached. Check that it is running and try again."
              : problem.message);
          }
        } finally {
          if (pending === request) {
            pending = null;
            setBusy(false);
          }
        }
      });

      paint([]);
      setBusy(false);
    })();
  </script>
</body>
</html>
'''

## Serve the pipeline

The server listens on a loopback port in the Colab runtime. The browser sends requests to the same server, and Python attaches the OpenRouter key.

In [51]:
import traceback
from http import HTTPStatus
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from threading import Lock, Thread


classifier_lock = Lock()


class QwenReviewError(RuntimeError):
    """OpenRouter failed to provide a valid review."""


def analyze_text(text: str, *, client: httpx.Client) -> str:
    """Run the classifier and return Qwen's validated JSON."""
    with classifier_lock:
        labels = classifier_top_three(text)

    try:
        return judge_emotions(text, labels, client=client)
    except (httpx.HTTPError, RuntimeError) as exc:
        raise QwenReviewError(str(exc)) from exc


def read_openrouter_key() -> str:
    key = os.environ.get("OPENROUTER_API_KEY", "").strip()
    if key:
        return key

    from google.colab import userdata

    try:
        key = userdata.get("OPENROUTER_API_KEY").strip()
    except (
        userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException
    ):
        key = ""

    if not key:
        key = getpass("OpenRouter API key (hidden input) ").strip()
    if not key:
        raise RuntimeError("Set OPENROUTER_API_KEY or enter the key at the prompt.")
    return key


def emotion_request_handler(analyze):
    page = PAGE_HTML.encode("utf-8")

    class EmotionHandler(BaseHTTPRequestHandler):
        def log_message(self, format, *args):
            # The wheel displays request errors and the notebook logs unexpected failures.
            pass

        def send_body(self, status, body, content_type):
            self.send_response(status)
            self.send_header("Content-Type", content_type)
            self.send_header("Content-Length", str(len(body)))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            if self.command != "HEAD":
                self.wfile.write(body)

        def send_problem(self, status, message):
            body = json.dumps({"error": message}).encode("utf-8")
            self.send_body(status, body, "application/json; charset=utf-8")

        def do_GET(self):
            if self.path.partition("?")[0] != "/":
                self.send_problem(HTTPStatus.NOT_FOUND, "Page unavailable.")
                return
            self.send_body(HTTPStatus.OK, page, "text/html; charset=utf-8")

        do_HEAD = do_GET

        def do_POST(self):
            if self.path.partition("?")[0] != "/api/analyze":
                self.send_problem(HTTPStatus.NOT_FOUND, "Endpoint unavailable.")
                return
            if self.headers.get_content_type() != "application/json":
                self.send_problem(HTTPStatus.UNSUPPORTED_MEDIA_TYPE, "Send a JSON request.")
                return

            try:
                length = int(self.headers.get("Content-Length", "0"))
                if length <= 0:
                    raise ValueError("Expected a JSON request body.")
                data = json.loads(self.rfile.read(length).decode("utf-8"))
            except (ValueError, OverflowError):
                self.send_problem(HTTPStatus.BAD_REQUEST, "Send a valid JSON request body.")
                return

            text = data.get("text") if isinstance(data, dict) else None
            if not isinstance(text, str) or not text.strip():
                self.send_problem(HTTPStatus.BAD_REQUEST, "Enter text to analyze.")
                return

            try:
                result = analyze(text)
            except QwenReviewError as exc:
                self.send_problem(HTTPStatus.BAD_GATEWAY, f"Qwen review failed. {exc}")
                return
            except Exception:
                traceback.print_exc()
                self.send_problem(
                    HTTPStatus.INTERNAL_SERVER_ERROR,
                    "Analysis failed. Check the notebook output for details.",
                )
                return

            self.send_body(HTTPStatus.OK, result.encode("utf-8"), "application/json; charset=utf-8")

    return EmotionHandler


def start_emotion_server(api_key):
    def analyze_with_key(text):
        with httpx.Client(
            base_url=BASE_URL,
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=120.0,
        ) as client:
            return analyze_text(text, client=client)

    server = ThreadingHTTPServer(("127.0.0.1", 0), emotion_request_handler(analyze_with_key))
    Thread(target=server.serve_forever, name="emotion-wheel", daemon=True).start()
    return server

## Open the app in Colab

Run the cell below to start the server and open the app in its output. Keep the runtime connected while using the app. Rerun this cell to reopen the preview after loading the model. [Colab port preview](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py#L43)

In [52]:
from google.colab import output

new_emotion_server = start_emotion_server(read_openrouter_key())
if "emotion_server" in globals():
    emotion_server.shutdown()
    emotion_server.server_close()
emotion_server = new_emotion_server

output.serve_kernel_port_as_iframe(emotion_server.server_port, height=900)

<IPython.core.display.Javascript object>